# Exploratory analysis: Pima Indians Diabetes

This notebook inspects the **local** Pima Indians Diabetes CSV, then shows the
preprocessing ideas used by the training pipeline.

**Educational use only — not a medical diagnostic tool.** The table has 768 rows
and includes only **adult women (age 21+) of Pima heritage**. Do not treat any
plot or accuracy number as a clinical result.

The dataset is **not downloaded here**. If the CSV is missing, follow
`diabetes_ann/data/README.md`.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# Make the project importable whether you opened Jupyter from diabetes_ann/
# or from a notebooks/ subfolder.
here = Path.cwd().resolve()
root = here if (here / "src").exists() else here.parent
src = root / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from diabetes_prediction.config import (
    DISCLAIMER,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    ZERO_AS_MISSING_COLUMNS,
)
from diabetes_prediction.data import load_dataset, split_xy
from diabetes_prediction.plots import write_eda_plots
from diabetes_prediction.preprocess import ZeroAsMissing, build_pipeline
from diabetes_prediction.profile import profile_dataframe, profile_to_markdown
from diabetes_prediction.reproducibility import seed_everything
from sklearn.linear_model import LogisticRegression

seed_everything(42)
sns.set_theme(style="whitegrid")
print(DISCLAIMER)
print("Project root:", root)

## 1. Load the local CSV

`load_dataset()` reads a file already on disk. It never calls the network.

In [ ]:
df = load_dataset()
print(df.shape)
df.head()

## 2. Profile: balance, duplicates, missing values, zeros

Several columns use `0` as a missing-value **code**, not a real measurement.
`Pregnancies = 0` is different: it is a valid count.

In [ ]:
profile = profile_dataframe(df)
print("rows:", profile["n_rows"])
print("duplicates:", profile["duplicate_rows"])
print("class counts:", profile["class_counts"])
print("positive rate:", round(profile["positive_rate"], 3))
print("true NaN missing:", profile["true_nan_missing"])
print("raw zero counts:", profile["zero_counts"])
print("zeros recoded as missing:", profile["zeros_treated_as_missing"])
print("zeros kept:", profile["zeros_kept_as_valid"])
print("high missingness after recode:", profile["high_missingness_after_recode"])
print()
print(profile["pregnancies_note"])

## 3. Why some zeros are missing (and pregnancies are not)

In [ ]:
for column, reason in profile["zero_as_missing_justification"].items():
    print(f"{column}: {reason}")
    print()

## 4. Exploratory plots

Figures are also written under `reports/figures/` so they match the CLI run.

In [ ]:
figure_dir = root / "reports" / "figures"
paths = write_eda_plots(df, figure_dir)
for path in paths:
    print(path.relative_to(root))

In [ ]:
from IPython.display import Image, display, Markdown

for name, title in [
    ("outcome_balance.png", "Class balance"),
    ("zero_counts.png", "Raw zero counts (red = treated as missing)"),
    ("feature_distributions.png", "Feature distributions"),
    ("outliers_boxplots.png", "Outliers"),
    ("feature_correlations.png", "Correlations"),
]:
    display(Markdown(f"### {title}"))
    display(Image(filename=str(figure_dir / name)))

## 5. Recoding zeros inside a transformer

The same rule is applied later **inside each training fold**, not as a
one-off edit of the whole CSV. That keeps the experiment honest: the test
fold is transformed with a rule defined up front, not with statistics
learned from the test rows.

`Pregnancies` still contains zeros after the transform.

In [ ]:
raw_zeros = (df[FEATURE_COLUMNS] == 0).sum()
recoded = ZeroAsMissing().fit_transform(df[FEATURE_COLUMNS])
recoded_missing = recoded.isna().sum()

compare = pd.DataFrame({
    "raw_zeros": raw_zeros,
    "missing_after_recode": recoded_missing,
    "treated_as_missing": [c in ZERO_AS_MISSING_COLUMNS for c in FEATURE_COLUMNS],
})
compare

In [ ]:
print("Pregnancies zeros kept:", int((recoded["Pregnancies"] == 0).sum()))
print("Glucose zeros remaining:", int((recoded["Glucose"] == 0).sum()))
print("Glucose now missing:", int(recoded["Glucose"].isna().sum()))

## 6. The leakage-safe pipeline (preview)

Imputation, scaling, and optional feature selection live in one pipeline so
`fit` happens on the training fold only. The next cell shows the steps; the
full ANN vs logistic regression vs random forest comparison is:

```bash
cd diabetes_ann
PYTHONPATH=src python -m diabetes_prediction
```

That command writes `reports/model_comparison.md`. This notebook does not
invent those metrics.

In [ ]:
pipe = build_pipeline(
    LogisticRegression(max_iter=500),
    feature_selection=True,
    k=5,
    resample=False,
)
pipe

## 7. Models trained by the CLI (not in this notebook)

The comparison uses the **same stratified folds** and the **same preprocessing** for every model so the difference is the classifier, not a hidden data leak.

| Model | Beginner picture |
|-------|------------------|
| Dummy most-frequent | Always guess the majority class (a sanity floor, ~65% accuracy here) |
| Logistic regression | A linear baseline |
| Random forest | A tree-based baseline (`max_depth=6`) |
| MLP ANN | Small feed-forward net: **8 inputs → 16 ReLU → 8 ReLU → 1 logistic output** |

Feature selection (`SelectKBest`, ANOVA F-test) is an extra pipeline step. When it is on, it is **fit on the training fold only**, then applied to that fold's test rows.

This notebook stops at exploration. Metrics come from:

```bash
cd diabetes_ann
python run_experiment.py --seed 42 --splits 5 --select-k 5
```

## 8. Reproducibility and limits

- Random seed: `42` (see `diabetes_prediction.config.RANDOM_SEED`).
- Stratified 5-fold CV is the evaluation protocol in the CLI.
- Do not report performance until `reports/model_comparison.md` exists.
- This sample is small and historically specific. It is not a screening tool.

In [ ]:
X, y = split_xy(df)
print("X:", X.shape, "y positive rate:", float(y.mean()))
print(DISCLAIMER)